# Sections

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cmocean
from pyprojroot import here
from ruamel.yaml import YAML

from blomster.grid import read_strait_indices


In [ ]:
# Get project root
root_path = Path(here())
yaml=YAML()
with open(root_path.joinpath('notebooks/settings.yml'), "r") as f:
    cfg = yaml.load(f)

# info of diagnostic tool and generated webpages
work_root = cfg['blomster']['work_root']
www_root = cfg['jupyterbook']['www_root']

# info of experiments
caseid = cfg['experiment']['caseid']
input_root = cfg['experiment']['input_root']
grid_label = cfg['experiment']['grid_label']
year_start = cfg['experiment']['year_start']
year_end = cfg['experiment']['year_end']

# info of grids
grid_file = cfg['grids'][grid_label]['grid_file']
region_file = cfg["grids"][grid_label]["region_file"]
section_file = cfg["grids"][grid_label]["section_file"]
grid_file = cfg['grids'][grid_label]['grid_file']
mask_RECCAP2_blom_file = cfg['grids'][grid_label]['mask_RECCAP2_file']

# info of observational datasets
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
mask_RECCAP2_file = cfg['datasets']['PHC3.0']['mask_RECCAP2_file']

# info of model climatologies
climo_file1 = cfg['climos']['blom']['hy']['ANN']
climo_file2 = cfg['climos']['blom']['hm']['ANN']


In [ ]:
with xr.open_dataset(grid_file) as ds:
    pdepth = ds['pdepth']
    plat = ds['plat']
    plon = ds['plon']

# mapping of variable names from PHC to BLOM
varmappings = {'temp': 'templvl', 'salt': 'salnlvl'}
cmapmapping = {'temp': 'thermal', 'salt': 'haline'}
datasource = {'temp': 'thermal', 'salt': 'haline'}

# option 1: selective sections
section_names = ['fram_strait']

for section_name in section_names:
    iidx, jidx, _, _ = read_strait_indices(section_file, section_name)
#   k = 0   # order of plots in the subplots
    for varphc, varblom in varmappings.items():
        fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))
        # read BLOM data and compute profile
        try:
            ds = xr.open_dataset(climo_file1)
        except (FileNotFoundError):
            try:
                ds = xr.open_dataset(climo_file2)
            except FileNotFoundError:
                print(f"{climo_file2} not found either, stop...")
                raise

        da = ds[varblom].isel(time=0)
        section = da.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
       #print(section)
        seclat = plat.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
        seclon = plon.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )
        secdepth = pdepth.isel(
             y=xr.DataArray(jidx, dims="points"),
             x=xr.DataArray(iidx, dims="points")
        )

        # set points with lat/lon values, and find the maxium depth
        section['points'] = seclon
        botdepth = secdepth.max().values

        # read PHC data and compute profile
        with xr.open_dataset(phc_file) as ds:
            da_phc = ds[varphc]
            da_phc = da_phc.assign_coords(lon=(((da_phc.lon + 180) % 360) - 180)).sortby("lon")
            section_phc = da_phc.sel(lat=78.5, lon=slice(-9,17))

        # get min, max values within certain quantile
        vals = np.concatenate([section.values.ravel(), section_phc.values.ravel()])
        vmin, vmax = np.nanquantile(vals, [0.001, 0.999])

        # plot the profiles
        section_phc.plot(ax=axs[0], x="lon", y="depth", ylim=[0, botdepth], vmin=vmin, vmax=vmax, cmap=f"cmo.{cmapmapping[varphc]}")
        section.plot(ax=axs[1], x="points", y="depth", ylim = [0, botdepth], vmin=vmin, vmax=vmax, cmap=f"cmo.{cmapmapping[varphc]}")

        axs[0].set_title(f'PHC3.0: {section_phc.standard_name} across {section_name}')
        axs[1].set_title(f'NorESM: {section.standard_name} across {section_name}')
        axs[0].invert_yaxis()
        axs[1].invert_yaxis()

    plt.show()
